# 3.1 — Multi-Dataset Regression Modelling (2.1 Manual vs 2.2 UMAP/MICE)

Run the **same regression pipeline** across multiple prepared datasets and find the best-performing **(dataset, model)** combination.

Datasets compared:
- **2.1-Manual** → `2.1-train.parquet` / `2.1-test.parquet`
- **2.2-UMAP-MICE** → `2.2-train.parquet` / `2.2-test.parquet`

## Outline

| # | Section |
|---|---|
| 1 | Imports and global setup |
| 2 | Reusable pipeline factory function |
| 3 | Load all datasets |
| 4 | Run pipeline on every dataset |
| 5 | Per-dataset CV model comparison |
| 6 | Cross-dataset test-set comparison (winner table) |
| 7 | Residual analysis — best pipeline |
| 8 | Feature importance & SHAP — best pipeline |
| 9 | Dummy patient prediction — best pipeline |
| 10 | Save all models and metrics |


## 1. Imports and Global Setup

In [2]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import polars as pl

from sklearn.model_selection import RepeatedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

# Optional explainability library
try:
    import shap
except Exception:
    shap = None

pd.options.display.max_columns = 200
pd.options.display.precision = 4

print('Libraries loaded. SHAP available:', shap is not None)


Libraries loaded. SHAP available: True


## 2. Reusable Pipeline Factory Function

`build_regression_pipeline` encapsulates the full modelling workflow:

1. Feature / target separation  
2. Numeric & categorical column detection  
3. `ColumnTransformer` preprocessor (imputation → scaling / one-hot)  
4. Model dictionary (linear + tree-based)  
5. `RepeatedKFold` cross-validation (MAE / RMSE / R²)  
6. Returns everything needed for downstream evaluation & explainability

**To switch to a different dataset** just call the function again with different `train_df` / `test_df` arguments.


In [ ]:
def build_regression_pipeline(
    train_df: pl.DataFrame,
    test_df:  pl.DataFrame,
    target: str,
    drop_cols: list = None,
    n_splits: int = 5,
    n_repeats: int = 3,
    random_state: int = 42,
):
    """
    Build, cross-validate, and return a full sklearn regression pipeline.

    Parameters
    ----------
    train_df : pl.DataFrame  — Polars training frame
    test_df  : pl.DataFrame  — Polars test frame
    target   : str           — regression target column name
    drop_cols: list          — extra columns to exclude from features
    n_splits, n_repeats, random_state — RepeatedKFold settings

    Returns
    -------
    cv_results_df  : pd.DataFrame
    best_pipeline  : fitted sklearn Pipeline
    X_train, X_test: pl.DataFrame  (Polars, feature columns only)
    y_train, y_test: np.ndarray
    feature_names  : list[str]  (after one-hot expansion)
    numeric_cols, cat_cols : list[str]  (raw Polars column names)
    models         : dict of candidate estimators
    """
    # ── 1. Feature / target separation (Polars) ──────────────────────────────
    all_drop     = [c for c in ([target] + (drop_cols or [])) if c in train_df.columns]
    feature_cols = [c for c in train_df.columns if c not in all_drop]

    X_train = train_df.select(feature_cols)
    y_train = train_df[target].to_numpy()
    X_test  = test_df.select([c for c in feature_cols if c in test_df.columns])
    y_test  = test_df[target].to_numpy()

    # ── 2. Column type detection using Polars native dtypes ──────────────────
    _numeric_dtypes = {
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        pl.Float32, pl.Float64,
    }
    _string_dtypes = {pl.Utf8, pl.String, pl.Categorical}

    numeric_cols = [c for c in X_train.columns if X_train[c].dtype in _numeric_dtypes]
    cat_cols     = [c for c in X_train.columns if X_train[c].dtype in _string_dtypes]

    print(f'Numeric features     : {len(numeric_cols)}')
    print(f'Categorical features : {len(cat_cols)}')
    print(f'  numeric dtypes     : {sorted({str(X_train[c].dtype) for c in numeric_cols})}')

    # ── 3. Convert to pandas only at the sklearn boundary ────────────────────
    X_train_pd = X_train.to_pandas()
    X_test_pd  = X_test.to_pandas()

    # ── 4. ColumnTransformer preprocessor ────────────────────────────────────
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    transformers = []
    if numeric_cols:
        transformers.append(('num', numeric_transformer, numeric_cols))
    if cat_cols:
        categorical_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ])
        transformers.append(('cat', categorical_transformer, cat_cols))

    if not transformers:
        raise ValueError('No numeric or categorical columns found.')

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

    # ── 5. Candidate models ───────────────────────────────────────────────────
    models = {
        'LinearRegression': LinearRegression(),
        'Ridge':            Ridge(alpha=1.0, random_state=random_state),
        'Lasso':            Lasso(alpha=0.01, random_state=random_state, max_iter=10_000),
        'RandomForest':     RandomForestRegressor(n_estimators=200, random_state=random_state, n_jobs=-1),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=300, random_state=random_state),
    }

    # ── 6. RepeatedKFold cross-validation ────────────────────────────────────
    cv      = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)
    scoring = {
        'MAE':  'neg_mean_absolute_error',
        'RMSE': 'neg_root_mean_squared_error',
        'R2':   'r2',
    }

    results = []
    for name, estimator in models.items():
        pipe   = Pipeline([('pre', preprocessor), ('model', estimator)])
        cv_res = cross_validate(pipe, X_train_pd, y_train, cv=cv, scoring=scoring,
                                n_jobs=-1, return_train_score=False)
        results.append({
            'model': name,
            'MAE':  -cv_res['test_MAE'].mean(),
            'RMSE': -cv_res['test_RMSE'].mean(),
            'R2':    cv_res['test_R2'].mean(),
        })
        print(f'  {name:<20} RMSE={results[-1]["RMSE"]:.4f}  R²={results[-1]["R2"]:.4f}')

    cv_results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)

    # ── 7. Fit best model on full training set ────────────────────────────────
    best_name      = cv_results_df.iloc[0]['model']
    best_estimator = models[best_name]
    best_pipeline  = Pipeline([('pre', preprocessor), ('model', best_estimator)])
    best_pipeline.fit(X_train_pd, y_train)
    print(f'\nBest model (CV): {best_name}')

    # ── 8. Reconstruct feature names after preprocessing ─────────────────────
    onehot_cols = []
    if cat_cols:
        enc = best_pipeline.named_steps['pre'].named_transformers_['cat'].named_steps['onehot']
        onehot_cols = enc.get_feature_names_out(cat_cols).tolist()
    feature_names = numeric_cols + onehot_cols

    # X_train / X_test returned as Polars for downstream Polars-native ops;
    # y_train / y_test returned as numpy for sklearn / arithmetic.
    return cv_results_df, best_pipeline, X_train, X_test, y_train, y_test, feature_names, numeric_cols, cat_cols, models

print('build_regression_pipeline() defined.')


build_regression_pipeline() defined.


## 3. Load All Datasets

Add or remove entries in `DATASETS` to compare more pipelines.  
Each entry is `label → {train_path, test_path}`.


In [ ]:
# ── Dataset registry — add/remove entries freely ─────────────────────────────
DATASETS = {
    '2.1-Manual': {
        'train_path': './data/interim/2.1-train.parquet',
        'test_path':  './data/interim/2.1-test.parquet',
    },
    '2.2-UMAP-MICE': {
        'train_path': './data/interim/2.2-train.parquet',
        'test_path':  './data/interim/2.2-test.parquet',
    },
}

TARGET = 'health_gain'

# ── Load each dataset as Polars DataFrames (no pandas conversion here) ────────
all_data = {}   # label -> (train_pl, test_pl, drop_cols)

for label, paths in DATASETS.items():
    train_pl  = pl.read_parquet(paths['train_path'])
    test_pl   = pl.read_parquet(paths['test_path'])
    drop_cols = ['OHS_Success'] if 'OHS_Success' in train_pl.columns else []
    all_data[label] = (train_pl, test_pl, drop_cols)
    print(f'{label:20s}  train={train_pl.shape}  test={test_pl.shape}  extra_drop={drop_cols}')

print(f'\nTarget: {TARGET}')
print(f'Datasets loaded: {list(all_data.keys())}')


Train shape: (94550, 46)
Test shape:  (23638, 46)
Columns: ['Provider Code', 'Procedure', 'Revision Flag', 'Year', 'Age Band', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System']


,Provider Code,Procedure,Revision Flag,Year,Age Band,Gender,Pre-Op Q Assisted,Pre-Op Q Assisted By,Pre-Op Q Symptom Period,Pre-Op Q Previous Surgery,Pre-Op Q Living Arrangements,Pre-Op Q Disability,Heart Disease,High Bp,Stroke,Circulation,Lung Disease,Diabetes,Kidney Disease,Nervous System,Liver Disease,Cancer,Depression,Arthritis,Pre-Op Q Mobility,Pre-Op Q Self-Care,Pre-Op Q Activity,Pre-Op Q Discomfort,Pre-Op Q Anxiety,Pre-Op Q EQ5D Index Profile,Pre-Op Q EQ5D Index,Post-Op Q EQ5D Index Profile,Knee Replacement Pre-Op Q Pain,Knee Replacement Pre-Op Q Night Pain,Knee Replacement Pre-Op Q Washing,Knee Replacement Pre-Op Q Transport,Knee Replacement Pre-Op Q Walking,Knee Replacement Pre-Op Q Standing,Knee Replacement Pre-Op Q Limping,Knee Replacement Pre-Op Q Kneeling,Knee Replacement Pre-Op Q Work,Knee Replacement Pre-Op Q Confidence,Knee Replacement Pre-Op Q Shopping,Knee Replacement Pre-Op Q Stairs,Knee Replacement Pre-Op Q Score,health_gain
0,268,1,0,3,5,2.0,2,1,2,2,2,1,1,0,0,0,1,0,1,0,0,0,0,1,2,2,2,3,2,22232,-0.016,22222,0,0,3,1,1,1,0,0,0,1,0,0,7.0,16.0
1,69,1,0,2,4,1.0,2,1,2,2,1,2,0,0,0,0,0,0,0,0,0,0,0,1,2,1,2,2,2,21222,0.620,11222,1,4,3,3,0,2,1,2,2,3,3,2,26.0,12.0
2,267,1,0,2,2,2.0,2,1,4,2,1,2,0,1,0,0,0,0,0,0,0,0,0,1,2,2,2,2,1,22221,0.587,21221,1,2,3,2,2,2,1,1,1,2,1,2,20.0,8.0
3,236,1,0,3,4,1.0,2,1,2,2,1,1,0,1,0,0,0,0,0,0,0,0,0,1,2,1,2,2,1,21221,0.691,11111,1,4,4,4,2,3,0,1,2,0,3,2,26.0,17.0
4,263,1,0,2,4,2.0,2,1,3,2,1,2,0,0,0,0,0,0,0,0,0,0,0,1,2,1,2,2,2,21222,0.620,11111,1,0,2,2,3,2,2,2,2,3,3,3,25.0,20.0



Train columns with nulls:
Series([], dtype: int64)

Target column  : health_gain
Columns to drop (besides target): []


## 4. Run Pipeline on Every Dataset

One call to `build_regression_pipeline` per dataset.  
Results (CV scores, test metrics, fitted pipeline) are stored in `all_results`.


In [ ]:
all_results = {}   # label -> dict of pipeline outputs + test metrics

for label, (train_pl, test_pl, drop_cols) in all_data.items():
    print(f'\n{"=" * 65}')
    print(f'  Dataset: {label}')
    print(f'{"=" * 65}')

    (
        cv_results_df,
        best_pipeline,
        X_train, X_test,       # pl.DataFrame
        y_train, y_test,       # np.ndarray
        feature_names,
        numeric_cols,
        cat_cols,
        models,
    ) = build_regression_pipeline(
        train_pl, test_pl, target=TARGET, drop_cols=drop_cols
    )

    best_name = cv_results_df.iloc[0]['model']
    # sklearn predict requires pandas; convert at the boundary
    y_pred    = best_pipeline.predict(X_test.to_pandas())
    mae       = mean_absolute_error(y_test, y_pred)
    rmse      = np.sqrt(mean_squared_error(y_test, y_pred))
    r2        = r2_score(y_test, y_pred)

    all_results[label] = {
        'cv_results_df': cv_results_df,
        'best_pipeline': best_pipeline,
        'best_name':     best_name,
        'X_train': X_train, 'X_test':  X_test,   # pl.DataFrame
        'y_train': y_train, 'y_test':  y_test,   # np.ndarray
        'y_pred':  y_pred,                        # np.ndarray
        'feature_names': feature_names,
        'numeric_cols':  numeric_cols,
        'cat_cols':      cat_cols,
        'models':        models,
        'test_mae':  mae,
        'test_rmse': rmse,
        'test_r2':   r2,
    }

    print(f'\n  >> Best CV model : {best_name}')
    print(f'     Test  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}')

print('\n\nAll pipelines complete.')


## 5. Per-Dataset CV Model Comparison

For each dataset, show a grouped bar chart (MAE / RMSE / R²) across all candidate models.  
The best model per dataset is highlighted in gold.


In [ ]:
cv_metrics = ['MAE', 'RMSE', 'R2']

for label, res in all_results.items():
    cv_df     = res['cv_results_df']
    best_name = res['best_name']

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'CV Model Comparison — {label}', fontsize=13)

    for ax, metric in zip(axes, cv_metrics):
        data   = cv_df.sort_values(metric, ascending=(metric != 'R2'))
        colors = ['gold' if m == best_name else 'steelblue' for m in data['model']]
        sns.barplot(ax=ax, data=data, x='model', y=metric, palette=colors)
        ax.set_title(f'CV {metric}')
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=35)

    plt.tight_layout()
    plt.show()

    print(f'  Best model ({label}): {best_name}  (highlighted in gold)')
    display(cv_df.reset_index(drop=True))
    print()


## 6. Cross-Dataset Test-Set Comparison — Winner Table

Compare every **(dataset, best model)** pair on the held-out test set.  
The row with the lowest RMSE is the overall winner.


In [ ]:
# ── Build comparison table (one row per dataset) ────────────────────────────
comparison_rows = []
for label, res in all_results.items():
    comparison_rows.append({
        'Dataset':    label,
        'Best Model': res['best_name'],
        'Test MAE':   round(res['test_mae'],  4),
        'Test RMSE':  round(res['test_rmse'], 4),
        'Test R²':    round(res['test_r2'],   4),
    })

comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values('Test RMSE')
    .reset_index(drop=True)
)

# Highlight winning row
def highlight_winner(s):
    return ['background-color: #d4edda; font-weight: bold'
            if i == 0 else '' for i in range(len(s))]

print('=== Cross-Dataset Test-Set Comparison ===\n')
display(comparison_df.style.apply(highlight_winner, axis=0))

# ── Announce the winner ──────────────────────────────────────────────────────
winner_row   = comparison_df.iloc[0]
best_dataset = winner_row['Dataset']
best_model   = winner_row['Best Model']

print(f'\n>>> WINNER')
print(f'    Dataset : {best_dataset}')
print(f'    Model   : {best_model}')
print(f'    RMSE    : {winner_row["Test RMSE"]}  |  R² : {winner_row["Test R²"]}')

# ── Side-by-side bar chart: all datasets × metrics ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Cross-Dataset Test-Set Performance', fontsize=13)

for ax, metric in zip(axes, ['Test MAE', 'Test RMSE', 'Test R²']):
    colors = ['gold' if d == best_dataset else 'steelblue'
              for d in comparison_df['Dataset']]
    sns.barplot(ax=ax, data=comparison_df, x='Dataset', y=metric, palette=colors)
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

# ── Grab the winning pipeline's objects for downstream cells ─────────────────
best_res      = all_results[best_dataset]
best_pipeline = best_res['best_pipeline']
best_name     = best_res['best_name']
X_train       = best_res['X_train']
X_test        = best_res['X_test']
y_train       = best_res['y_train']
y_test        = best_res['y_test']
y_pred        = best_res['y_pred']
feature_names = best_res['feature_names']
numeric_cols  = best_res['numeric_cols']
cat_cols      = best_res['cat_cols']

print(f'\nDownstream sections will use: dataset={best_dataset}, model={best_name}')


## 7. Residual Analysis — Best Pipeline

Using the winning **(dataset, model)** pair identified above.


In [ ]:
resid = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Residual Analysis — {best_dataset}  /  {best_name}', fontsize=12)

sns.scatterplot(ax=axes[0], x=y_pred, y=resid, alpha=0.4)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Predicted vs Residual')

axes[1].hist(resid, bins=40, edgecolor='k', alpha=0.6)
axes[1].set_title('Residual distribution')
axes[1].set_xlabel('Residual')

plt.tight_layout()
plt.show()

print(f'Residual stats: mean={resid.mean():.4f}  std={resid.std():.4f}')


## 8. Feature Importance & SHAP — Best Pipeline


In [ ]:
model_step   = best_pipeline.named_steps['model']
title_suffix = f'{best_dataset}  /  {best_name}'

# ── Tree models: feature importances ─────────────────────────────────────────
if hasattr(model_step, 'feature_importances_'):
    importances = (
        pd.Series(model_step.feature_importances_, index=feature_names)
        .sort_values(ascending=False)
    )
    print('Top 20 feature importances:')
    display(importances.head(20))

    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances.head(20).values, y=importances.head(20).index)
    plt.title(f'Top 20 Feature Importances — {title_suffix}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

# ── Linear models: coefficients ───────────────────────────────────────────────
elif hasattr(model_step, 'coef_'):
    coef     = pd.Series(model_step.coef_, index=feature_names)
    coef_abs = coef.abs().sort_values(ascending=False)
    top25    = coef.loc[coef_abs.head(25).index].sort_values()

    print('Top 25 features by absolute coefficient:')
    display(coef.loc[coef_abs.head(25).index])

    colors = ['tomato' if v < 0 else 'steelblue' for v in top25]
    plt.figure(figsize=(10, 8))
    sns.barplot(x=top25.values, y=top25.index, palette=colors)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title(f'Top 25 Feature Coefficients — {title_suffix}')
    plt.xlabel('Coefficient value')
    plt.tight_layout()
    plt.show()

else:
    print(f'No feature importance or coefficients for {best_name}.')

# ── SHAP — sklearn transform requires pandas ──────────────────────────────────
if shap is not None:
    print('\nRunning SHAP (may take a moment)...')
    X_train_t   = best_pipeline.named_steps['pre'].transform(X_train.to_pandas())
    X_test_t    = best_pipeline.named_steps['pre'].transform(X_test.to_pandas())
    explainer   = shap.Explainer(model_step, X_train_t)
    shap_values = explainer(X_test_t)

    shap.summary_plot(shap_values, feature_names=feature_names, plot_type='bar', show=True)
    shap.summary_plot(shap_values, feature_names=feature_names, show=True)
else:
    print('SHAP not available — restart kernel and re-run to activate.')


## 9. Dummy Patient Prediction — Best Pipeline

Simulate a median/mode patient from the winning dataset's training data, predict their health gain, and explain the key drivers.


In [ ]:
print(f'Dummy patient built from training set: {best_dataset}  /  {best_name}\n')

# ── Build representative patient using Polars native operations ───────────────
dummy_data = {}
for col in numeric_cols:
    dummy_data[col] = [X_train[col].median()]          # Polars returns a scalar
for col in cat_cols:
    mode_series = X_train[col].mode()
    dummy_data[col] = [mode_series[0] if len(mode_series) > 0 else None]

# Keep as Polars DataFrame for consistency.
# Convert to pandas only at the sklearn boundary.
dummy_pl = pl.DataFrame(dummy_data)
dummy_pd = dummy_pl.to_pandas()      # sklearn boundary

print('=== Dummy Patient Profile ===')
display(dummy_pl.transpose(include_header=True, header_name='Feature', column_names=['Value']))

# ── Predict (sklearn boundary) ────────────────────────────────────────────────
predicted_gain = best_pipeline.predict(dummy_pd)[0]
avg_gain       = float(np.mean(y_train))
diff           = predicted_gain - avg_gain
direction      = 'ABOVE' if diff > 0 else 'BELOW'

print(f'\n>>> Predicted Health Gain : {predicted_gain:.4f}')
print(f'    Training set mean     : {avg_gain:.4f}')
print(f'    This patient is {direction} average by {abs(diff):.4f} points\n')

# ── Explain contributions ─────────────────────────────────────────────────────
model_step = best_pipeline.named_steps['model']

if hasattr(model_step, 'coef_'):
    transformed   = best_pipeline.named_steps['pre'].transform(dummy_pd)[0]
    contributions = pd.Series(model_step.coef_ * transformed, index=feature_names)
    top_pos = contributions.sort_values(ascending=False).head(5)
    top_neg = contributions.sort_values().head(5)

    print('=== Top 5 features INCREASING the predicted health gain ===')
    for feat, val in top_pos.items():
        raw = dummy_pl[feat][0] if feat in dummy_pl.columns else 'encoded'
        print(f'  + {feat:<45} raw={raw!r:<10}  contribution={val:+.4f}')

    print('\n=== Top 5 features DECREASING the predicted health gain ===')
    for feat, val in top_neg.items():
        raw = dummy_pl[feat][0] if feat in dummy_pl.columns else 'encoded'
        print(f'  - {feat:<45} raw={raw!r:<10}  contribution={val:+.4f}')

    sorted_contrib = contributions.abs().sort_values(ascending=False).head(20)
    colors = ['steelblue' if contributions[f] >= 0 else 'tomato' for f in sorted_contrib.index]
    plt.figure(figsize=(10, 5))
    sns.barplot(x=contributions[sorted_contrib.index].values,
                y=sorted_contrib.index, palette=colors)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title(f'Feature Contributions — {best_dataset}  /  {best_name}')
    plt.xlabel('Contribution  (coef × scaled value)')
    plt.tight_layout()
    plt.show()

elif hasattr(model_step, 'feature_importances_'):
    importances = pd.Series(model_step.feature_importances_, index=feature_names).sort_values(ascending=False)
    print('=== Top 10 most important features ===')
    for feat, imp in importances.head(10).items():
        raw = dummy_pl[feat][0] if feat in dummy_pl.columns else 'encoded'
        print(f'  {feat:<45} patient_value={raw!r:<10}  importance={imp:.4f}')

# ── SHAP waterfall (sklearn boundary: .to_pandas()) ───────────────────────────
if shap is not None:
    X_train_t = best_pipeline.named_steps['pre'].transform(X_train.to_pandas())
    X_dummy_t = best_pipeline.named_steps['pre'].transform(dummy_pd)
    explainer = shap.Explainer(model_step, X_train_t)
    shap_vals = explainer(X_dummy_t)
    print('\n=== SHAP Waterfall (dummy patient) ===')
    shap.plots.waterfall(shap_vals[0], max_display=15, show=True)
else:
    print('\nTip: restart kernel and re-run to enable SHAP waterfall.')

print('\n=== Plain English Summary ===')
print(f"Dataset '{best_dataset}' with model '{best_name}' predicts this patient will")
print(f"gain {predicted_gain:.2f} points on the knee health score after surgery.")
print(f"This is {'higher' if diff > 0 else 'lower'} than the average gain of {avg_gain:.2f} in that training set.")


## 10. Save All Models and Metrics

Save the best pipeline for **each dataset** and a combined comparison CSV.


In [ ]:
out_dir = Path('./models')
out_dir.mkdir(parents=True, exist_ok=True)

# ── Save best pipeline for every dataset ─────────────────────────────────────
for label, res in all_results.items():
    safe_label = label.replace('/', '-').replace(' ', '_')
    pipe_path  = out_dir / f'3.1-{safe_label}-{res["best_name"]}.joblib'
    joblib.dump(res['best_pipeline'], pipe_path)
    print(f'Saved model  [{label}]: {pipe_path}')

# ── Save cross-dataset comparison CSV ────────────────────────────────────────
comparison_path = out_dir / '3.1-comparison-metrics.csv'
comparison_df.to_csv(comparison_path, index=False)
print(f'\nSaved comparison table : {comparison_path}')

# ── Print final winner ────────────────────────────────────────────────────────
print(f'\n{"=" * 50}')
print(f'  OVERALL WINNER')
print(f'  Dataset : {best_dataset}')
print(f'  Model   : {best_name}')
print(f'  RMSE    : {winner_row["Test RMSE"]}  |  R² : {winner_row["Test R²"]}')
print(f'{"=" * 50}')
